In [ ]:

# --------------------------------------------------
# 3. Decision Tree from scratch + Manual GridSearch
# --------------------------------------------------

def impurity(y, criterion="gini"):
    if len(y) == 0:
        return 0.0

    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)

    if criterion == "gini":
        return 1.0 - np.sum(probabilities ** 2)

    # Entropy
    return -np.sum(
        probabilities * np.log2(probabilities + 1e-12)
    )


def best_split(X, y, criterion="gini", min_samples_split=2):
    best_gain = 0.0
    best_info = None
    parent_impurity = impurity(y, criterion)

    if len(y) < min_samples_split:
        return None, 0.0

    for feature in range(X.shape[1]):
        values = np.unique(X[:, feature])

        if len(values) < 2:
            continue

        thresholds = (values[:-1] + values[1:]) / 2

        for threshold in thresholds:
            left = X[:, feature] <= threshold
            right = ~left

            if left.sum() == 0 or right.sum() == 0:
                continue

            weighted_impurity = (
                left.sum() / len(y) * impurity(y[left], criterion)
                + right.sum() / len(y) * impurity(y[right], criterion)
            )

            gain = parent_impurity - weighted_impurity

            if gain > best_gain:
                best_gain = gain
                best_info = (feature, threshold, left, right)

    return best_info, best_gain


def build_tree(
    X,
    y,
    depth=0,
    max_depth=4,
    criterion="gini",
    min_samples_split=10
):
    # Stop if the node is pure or too small to split
    if (
        len(np.unique(y)) == 1
        or depth >= max_depth
        or len(y) < min_samples_split
    ):
        return {"prediction": int(np.bincount(y).argmax())}

    split, gain = best_split(
        X, y, criterion, min_samples_split
    )

    if split is None:
        return {"prediction": int(np.bincount(y).argmax())}

    feature, threshold, left, right = split

    return {
        "feature": feature,
        "threshold": threshold,
        "left": build_tree(
            X[left], y[left], depth + 1, max_depth,
            criterion, min_samples_split
        ),
        "right": build_tree(
            X[right], y[right], depth + 1, max_depth,
            criterion, min_samples_split
        )
    }


def predict_one(row, node):
    if "prediction" in node:
        return node["prediction"]

    if row[node["feature"]] <= node["threshold"]:
        return predict_one(row, node["left"])

    return predict_one(row, node["right"])


def predict_tree(X, tree):
    return np.array([predict_one(row, tree) for row in X])


def accuracy_score_manual(y_true, predictions):
    return np.mean(y_true == predictions)


In [ ]:


# --------------------------------------------------
# 4. Create a validation split from training data
# --------------------------------------------------

rng = np.random.default_rng(42)
train_indices = []
validation_indices = []

# Stratify by target class
for class_value in np.unique(y_train):
    class_indices = np.where(y_train == class_value)[0]
    rng.shuffle(class_indices)

    split_point = int(0.8 * len(class_indices))

    train_indices.extend(class_indices[:split_point])
    validation_indices.extend(class_indices[split_point:])

train_indices = np.array(train_indices, dtype=int)
validation_indices = np.array(validation_indices, dtype=int)

X_fit = X_train[train_indices]
y_fit = y_train[train_indices]

X_val = X_train[validation_indices]
y_val = y_train[validation_indices]


In [ ]:
# --------------------------------------------------
# 5. Manual GridSearch
# --------------------------------------------------

depth_options = [2, 3, 4, 5, 6]
criterion_options = ["gini", "entropy"]
min_split_options = [2, 10, 20, 30]

results = []
best_score = -1
best_params = None
best_tree = None

for depth in depth_options:
    for criterion in criterion_options:
        for min_samples in min_split_options:

            candidate_tree = build_tree(
                X_fit,
                y_fit,
                max_depth=depth,
                criterion=criterion,
                min_samples_split=min_samples
            )

            val_predictions = predict_tree(X_val, candidate_tree)
            val_accuracy = accuracy_score_manual(
                y_val, val_predictions
            )

            results.append({
                "max_depth": depth,
                "criterion": criterion,
                "min_samples_split": min_samples,
                "validation_accuracy": val_accuracy
            })

            if val_accuracy > best_score:
                best_score = val_accuracy
                best_params = {
                    "max_depth": depth,
                    "criterion": criterion,
                    "min_samples_split": min_samples
                }
                best_tree = candidate_tree

# Show all combinations
results.sort(
    key=lambda row: row["validation_accuracy"],
    reverse=True
)

print("\nMANUAL GRID SEARCH RESULTS")
print("-" * 75)

for result in results:
    print(
        f"depth={result['max_depth']}, "
        f"criterion={result['criterion']:<7}, "
        f"min_split={result['min_samples_split']:>2}, "
        f"validation_accuracy="
        f"{result['validation_accuracy']:.4f}"
    )

print("\nBEST HYPERPARAMETERS")
print(best_params)
print(f"Best validation accuracy: {best_score:.4f}")




In [ ]:
best_tree = build_tree(
    X_train,
    y_train,
    max_depth=best_params["max_depth"],
    criterion=best_params["criterion"],
    min_samples_split=best_params["min_samples_split"]
)

tree_predictions = predict_tree(X_test, best_tree)

print(
    "\nTUNED DECISION TREE TEST ACCURACY:",
    f"{accuracy_score_manual(y_test, tree_predictions):.4f}"
)
